# 桥接到 Kubernetes：为什么需要 Device Plugin

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rootsongjc/gpu-lab-notebooks/blob/main/labs/zh/lab-11-bridge-to-k8s.ipynb)

Colab 篇的收尾。用单机 GPU 的直觉，理解 Kubernetes 为什么默认看不到 GPU、Device Plugin 解决什么，然后进入 Kubernetes 部分。

本 notebook 配合教程章节 [桥接到 Kubernetes：为什么需要 Device Plugin](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-11-bridge-to-k8s/) 使用。运行前请确认 **Runtime → Change runtime type → T4 GPU**。

每个 cell 对应教程中的一步；Markdown 单元摘录了必要的解释，完整上下文请回到教程阅读。

In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total,compute_cap --format=csv

## 为什么这个问题重要

回顾一下你在这十个 Lab 里做的事：打开一个 cell、`import torch`、用 GPU，甚至用 `nvcc` 编译了自己的 kernel。这一切之所以“能用”，是因为**你自己就是这台机器的管理员和唯一用户**：你知道有 GPU、知道怎么访问它、不需要跟别人协商。

但 Kubernetes 是个**多租户、多节点**的平台：成百上千个 Pod 在争用资源，调度器要决定“这个 Pod 去哪个节点、能不能拿到 GPU、能拿多少”。原生 Kubernetes 只认 CPU/内存这种可分割资源，**它根本不认识 GPU**。本 Lab 用一个 Colab 能跑的小演示，让你先对这道鸿沟有个直观印象。

## 单机 vs 平台：同一个 GPU，两套问题



| 你在 Colab 关心的 | Kubernetes 平台要解决的 |
| :--- | :--- |
| 我的 tensor 有没有上 GPU？ | 这个 Pod 该分到哪个节点？ |
| 显存够不够我的模型？ | 节点上 `nvidia.com/gpu` 还剩多少？ |
| 利用率高不高？ | 谁在共享这张卡、配额怎么给？ |
| 用完 `empty_cache` | 隔离：别人会不会占用我的显存？ |

## 为什么 Kubernetes 看不到 GPU

[Lab 02](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-02-gpu-on-pcie/) 里你看到 GPU 挂在 PCIe `00:04.0` 上。但 Kubernetes 的调度器只看 `node.status.allocatable`，默认那里面**只有 cpu/memory/ephemeral-storage**，没有 GPU。

![Device Plugin 把物理 GPU 上报为 node.allocatable 里的 nvidia.com/gpu，调度器据此匹配 Pod 请求，共享方案再把一张卡切成多份并加配额](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-11-bridge-to-k8s/device-plugin-makes-gpu-visible.svg)

这就是 **Device Plugin** 存在的全部理由：在节点上发现 GPU，把它变成 `nvidia.com/gpu` 这个 Extended Resource 上报给 kubelet，让调度器“看见”。[Lab 06](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-06-nvml/) 你调的 NVML，就是 device plugin 用来发现设备的底层。

原生 Device Plugin 只能整卡分配（`nvidia.com/gpu: 1` 就是一整张）。而 [Lab 09](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-09-mps/)、[Lab 10](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-10-gpu-virtualization/) 告诉你，真实需求是共享加配额。Kubernetes 上有好几种方案补在 Device Plugin 之上：GPU Operator 的 time-slicing、MPS 和 MIG 模式，HAMi 的显存与算力配额，KAI Scheduler 的 GPU 分数，以及新一代资源 API DRA。它们把 [Lab 04](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-04-where-did-vram-go/) 的显存和 [Lab 05](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/colab/lab-05-gpu-utilization/) 的算力变成可调度的资源。整套技术栈的分层见 [Kubernetes GPU 技术栈全景](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/foundations/k8s-gpu-stack-landscape/)，Device Plugin 的局限和 DRA 的改进见 [Device Plugin 与 DRA 边界](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/foundations/device-plugin-vs-dra/)。

## 一个 Colab 能做的“桥接演示”

Colab 里没有 Kubernetes，但你可以模拟 Device Plugin 做的第一件事：**上报设备**，体会“把硬件翻译成资源”这一步：

In [ ]:
# 模拟 device plugin 对 kubelet 的上报：把 GPU 摘要成一条资源记录
!pip install -q nvidia-ml-py
from pynvml import *
nvmlInit()
h = nvmlDeviceGetHandleByIndex(0)
mem = nvmlDeviceGetMemoryInfo(h)
major, minor = nvmlDeviceGetCudaComputeCapability(h)
print("device report →")
print("  resource : nvidia.com/gpu")
print("  count    : 1")
print("  uuid     :", nvmlDeviceGetUUID(h))
print("  product  :", nvmlDeviceGetName(h))
print("  memory   :", round(mem.total/1024**2), "MiB")
print("  cc       :", f"{major}.{minor}")
print("  → kubelet 会把 count 写进 node.status.allocatable，GFD 会把 product/memory/cc 写成 node label")

实测输出（UUID 每张卡不同）：

```text
device report →
  resource : nvidia.com/gpu
  count    : 1
  uuid     : GPU-2174719e-76ca-d370-94ea-2132503039b7
  product  : Tesla T4
  memory   : 15360 MiB
  cc       : 7.5
  → kubelet 会把 count 写进 node.status.allocatable，GFD 会把 product/memory/cc 写成 node label
```

这几行在 Kubernetes 里各有去处：

- `resource` 和 `count`：Device Plugin 上报给 kubelet，最终出现在 `kubectl describe node` 的 `Allocatable: nvidia.com/gpu: 1`。Pod 写 `limits: nvidia.com/gpu: 1`，调度器就是拿它和这个数字比较；
- `uuid`：Device Plugin 内部用它区分每一张卡，Pod 被分到哪张卡、容器里注入哪个设备，都靠这个 ID；
- `product`、`memory`、`cc`：GPU Feature Discovery（GFD）把它们写成节点标签，比如 `nvidia.com/gpu.product=Tesla-T4`、`nvidia.com/gpu.memory=15360`。想让 Pod 只去 T4 节点，就用 `nodeSelector` 选这些标签。

注意原生 Device Plugin 上报的只有“几张卡”，没有“多少显存可分”。15360 MiB 只是标签，调度器不会拿它做加减，这正是共享方案和 DRA 要补的地方。

这就是 [本地 Fake GPU Playground](https://jimmysong.io/zh/book/gpu-lab-on-kubernetes/foundations/local-fake-gpu/) 里 fake-gpu-operator 在做的事，只不过它在一个 CPU 节点上**模拟**了这一步，让你没 GPU 也能验证控制面。

## 验证

完成后你应该能：

- 用一句话说清 Device Plugin 解决了什么（让 K8s 看见 GPU）；
- 解释为什么原生 K8s 默认不认识 GPU；
- 说出 Device Plugin 之上还有哪几类共享方案；

## 总结表：Colab 篇给你交付了什么



| 能力 | 来自 | Kubernetes 部分的对应 |
| :--- | :--- | :--- |
| 读懂 nvidia-smi | Lab 01–02 | 节点 GPU 上报、DCGM |
| CUDA 可用性 | Lab 03 | 驱动/runtime/CDI，镜像里的 CUDA 版本约束 |
| 显存动态 | Lab 04 | 共享时的显存配额与隔离 |
| 利用率语义 | Lab 05 | 算力配额、DCGM 监控 |
| NVML 设备信息 | Lab 06 | device plugin 发现、GFD label |
| CUDA 编程模型、nvcc、Compute Capability | Lab 07 | 镜像编译目标、节点选择（`nvidia.com/cuda.*` label） |
| Kernel 并行模型、CUDA Runtime API | Lab 08 | 软件共享方案拦截的层 |
| 共享机制边界 | Lab 09–10 | time-slicing、MPS、MIG、HAMi、KAI GPU 分数的选型 |
| 桥接 K8s | 本 Lab | Device Plugin |

## 跑完之后

如果你看到的输出与教程里的预期明显不符，或者碰到了教程没有列出的故障模式，欢迎带上驱动 / CUDA 版本和关键输出在 [rootsongjc/gpu-lab-notebooks](https://github.com/rootsongjc/gpu-lab-notebooks) 提 Issue 或 PR。